# 领航者7020：真实 PYNQ Ascon 验证

从仓库根目录启动 Jupyter，并按 `Restart Kernel and Run All Cells` 顺序执行。
本 Notebook 只使用真实 Overlay 和 release 固件，没有模拟回退。
运行结果会写入 `data/pynq/notebook_<时间>/`，便于直接归档和报告引用。

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import pynq

repo = Path(os.environ.get("ASCON_REPO", "/home/xilinx/jupyter_notebooks/amd_fpga_2026")).resolve()
assert (repo / "src/pynq/board_demo.py").is_file(), "请设置 ASCON_REPO 指向实际仓库"
os.chdir(str(repo))
sys.path.insert(0, str(repo))

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = repo / "data/pynq" / ("notebook_" + run_id)
run_dir.mkdir(parents=True, exist_ok=False)
identity = {
    "generated_local": datetime.now().isoformat(timespec="seconds"),
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=str(repo), text=True
    ).strip(),
    "pynq_version": pynq.__version__,
    "python_version": platform.python_version(),
    "kernel": platform.platform(),
}
(run_dir / "identity.json").write_text(
    json.dumps(identity, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(identity, indent=2))

## 1. 实板 KAT smoke

默认运行三模式各 6 个容量内代表 KAT，并检查错误 Tag 拒绝。
只有实际通过时才写最终 JSON；失败会立即终止 Notebook。

In [ ]:
import contextlib
import io
from src.pynq.board_demo import run

RUN_FULL_KAT = False
smoke_log = io.StringIO()
with contextlib.redirect_stdout(smoke_log), contextlib.redirect_stderr(smoke_log):
    smoke = run(
        "board/vivado_zynq7000_xc7z020_pynq/release",
        full_kat=RUN_FULL_KAT,
    )

print(smoke_log.getvalue())
assert smoke["board_validation"] == "PASS"
(run_dir / "pynq_board_smoke.json").write_text(
    json.dumps(smoke, indent=2, default=str) + "\n", encoding="utf-8"
)
(run_dir / "pynq_board_smoke.log").write_text(
    smoke_log.getvalue(), encoding="utf-8"
)
print("KAT evidence saved to", run_dir)

## 2. 固定输入 warm benchmark

使用官方 KAT `#529`：PT=16 bytes、AD=0。每种模式先预热 5 次，再记录 30 次
加密和 30 次解密。每次调用都检查 KAT，`mcycle` scope 不包含 PS/UART 传输。

In [ ]:
from src.pynq.board_benchmark import run_benchmark

benchmark_log = io.StringIO()
with contextlib.redirect_stdout(benchmark_log), contextlib.redirect_stderr(benchmark_log):
    benchmark = run_benchmark(
        "board/vivado_zynq7000_xc7z020_pynq/release",
        warmup=5,
        repetitions=30,
        kat_count=529,
    )

print(benchmark_log.getvalue())
assert benchmark["board_validation"] == "PASS"
(run_dir / "pynq_warm_benchmark.json").write_text(
    json.dumps(benchmark, indent=2) + "\n", encoding="utf-8"
)
(run_dir / "pynq_warm_benchmark.log").write_text(
    benchmark_log.getvalue(), encoding="utf-8"
)
print("benchmark evidence saved to", run_dir)

## 3. 结果摘要

cycles 用于 SW/HW/ISE 对比；wall time 只用于观察 PS 端到端开销，不能替代 `mcycle`。

In [ ]:
print("mode   op       median_cycles  p95_cycles  median_wall_ms")
for mode_name in ("sw", "hw", "ise"):
    for operation in ("encrypt", "decrypt"):
        item = benchmark["modes"][mode_name][operation]
        print("{:<6} {:<8} {:>13} {:>11} {:>15.3f}".format(
            mode_name,
            operation,
            item["summary"]["median"],
            item["summary"]["p95"],
            item["wall_ns_summary"]["median"] / 1_000_000,
        ))
print("speedup median vs SW:", benchmark["speedup_median_vs_sw"])

如需重新生成容量内全部 289 组 KAT，将 `RUN_FULL_KAT` 改为 `True` 后从第一个
代码单元重新执行。超过 16-byte 的 800 组仍属于 `NOT_RUN`，不能声明硬件全量通过。
